In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import os
import random
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(7)

In [ ]:
BASE_PATH = os.path.expanduser('~/Downloads/rock_csvs')

all_data = {
    'S10Granite': [
        os.path.join(BASE_PATH, 'S10Granite_1-83Hz_profiles.csv'),
        os.path.join(BASE_PATH, 'S10Granite_5-10Hz_profiles.csv'),
    ],
    'Holstein_Sandstone': [
        os.path.join(BASE_PATH, 'Holstein_Sandstone_1-83Hz_profiles.csv'),
        os.path.join(BASE_PATH, 'Holstein_Sandstone_5-10Hz_profiles.csv'),
    ],
    'Leitendorf_Limestone': [
        os.path.join(BASE_PATH, 'Leitendorf_Limestone_1-83Hz_profiles.csv'),
        os.path.join(BASE_PATH, 'Leitendorf_Limestone_5-10Hz_profiles.csv'),
    ],
}

def load_combined_data(data_dict):
    dfs = []
    class_names = list(data_dict.keys())

    for label, (rock_name, paths) in enumerate(data_dict.items()):
        rock_profiles = []
        for path in paths:
            df = pd.read_csv(path, header=None, decimal='.')
            df_numeric = df.iloc[3::6, :1060].astype(np.float32)
            rock_profiles.append(df_numeric.values)
            print(f'  {os.path.basename(path)}: {df_numeric.shape[0]} profiles')

        combined = np.vstack(rock_profiles)
        class_num = np.array([label] * combined.shape[0], dtype=np.float32)
        d = np.c_[combined, class_num]
        dfs.extend(d)
        print(f'{rock_name} total: {combined.shape[0]} profiles\n')

    dataset = np.array(dfs)
    return dataset, class_names

print('Loading combined dataset (1.83 Hz + 5.10 Hz)...')
dataset, class_names = load_combined_data(all_data)
print(f'Total combined dataset shape: {dataset.shape}')

In [ ]:
class_names

In [ ]:
X = dataset[:, :-1]
y = dataset[:, -1:]

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Classes: {np.unique(y)}')
print(f'Samples per class: {[(y == i).sum() for i in range(len(class_names))]}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
for i, (rock_name, label) in enumerate(zip(class_names, range(3))):
    idx = np.where(y.flatten() == label)[0][:5]
    for j in idx:
        axes[i].plot(X[j], alpha=0.5)
    axes[i].set_title(rock_name)
plt.tight_layout()
_RES = 'results_1d_cnn_combined_speed_csv'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'sample_profiles.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] sample_profiles.png -> {_p}')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size=0.2, random_state=3, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
class RockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __len__(self):
        return len(self.X)


train_dataset = RockDataset(X_train, y_train)
test_dataset  = RockDataset(X_test,  y_test)

In [ ]:
import itertools

batch_sizes   = [2048]
lrs           = [0.0001, 0.00001]
epochs_list   = [100]
weight_decays = [0., 1e-4]

hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({
        'batch_size'   : batch_size,
        'lr'           : lr,
        'epochs'       : epochs,
        'weight_decay' : weight_decay
    })

print(f'Generated {len(hyperparameters_list)} combinations.')

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_input = nn.LayerNorm(normalized_shape=1060)
        self.conv1 = nn.Conv1d(1, 16, kernel_size=9, stride=3)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=9, stride=3)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, stride=2)
        self.layer_norm_1 = nn.LayerNorm(normalized_shape=64)
        self.fnn1 = nn.Linear(64, len(class_names))

        self.relu        = nn.ReLU()
        self.max_pool    = nn.MaxPool1d(kernel_size=2)
        self.global_pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten     = nn.Flatten()
        self.dropout     = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.layer_norm_input(x)
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.max_pool(x)
        x = self.relu(self.conv2(x))
        x = self.max_pool(x)
        x = self.relu(self.conv3(x))
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.layer_norm_1(x)
        x = self.fnn1(x)
        return x


model = Model()
print(f'Model parameters: {sum(p.numel() for p in model.parameters())}')

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import json


def run_experiments(train_dataset, test_dataset, X_test, y_test):

    experiment_results = []
    results_dir = 'results_1d_cnn_combined_speed_csv'
    os.makedirs(results_dir, exist_ok=True)

    monitor_indices = np.arange(4)
    X_monitor = torch.from_numpy(X_test[monitor_indices]).to(device)
    y_monitor = torch.from_numpy(y_test[monitor_indices]).squeeze().long().to(device)

    print(f'Monitor True Labels: {[class_names[int(i)] for i in y_monitor]}')

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nStarting Experiment {idx+1}\nConfig: {config}\n{'='*20}")

        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
        test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, num_workers=0)

        model = Model().to(device)
        optimizer = torch.optim.Adam(params=model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')
        tb_log_dir      = os.path.join(results_dir, f'tb_exp{idx+1}_lr{config["lr"]}_wd{config["weight_decay"]}')
        writer          = SummaryWriter(log_dir=tb_log_dir)

        training_loss = []
        training_acc  = []
        testing_loss  = []
        testing_acc   = []

        for epoch in range(config['epochs']):
            avg_train_loss = []
            avg_train_acc  = []
            avg_test_loss  = []
            avg_test_acc   = []

            print(f'Epoch: {epoch + 1}/{config["epochs"]}')

            # Train
            model.train()
            for i, d in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                _, predicted_labels  = torch.max(outputs, 1)
                correct_predictions  = (predicted_labels == y_batch).sum().item()
                accuracy             = correct_predictions / y_batch.size(0)
                avg_train_loss.append(loss.item())
                avg_train_acc.append(accuracy)

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            # Test
            model.eval()
            all_preds, all_true = [], []
            with torch.no_grad():
                for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                    X_batch, y_batch = d
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.squeeze().long().to(device)
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)

                    preds = torch.max(outputs, 1)[1]
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(y_batch.cpu().numpy())

                    acc = (preds == y_batch).sum().item() / y_batch.size(0)
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append(acc)

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing: Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f} - Saved Model.')

            model.eval()
            with torch.no_grad():
                outputs_monitor = model(X_monitor)
                _, pred_monitor = torch.max(outputs_monitor, 1)
                print(f'Monitor Profiles (Epoch {epoch + 1}):')
                for j, (true_cls, pred_cls) in enumerate(zip(y_monitor, pred_monitor)):
                    status = '\u2713' if true_cls == pred_cls else '\u2717'
                    print(f'  Sample {monitor_indices[j]}: True: {class_names[true_cls.item()]:<15} | Pred: {class_names[pred_cls.item()]:<15} {status}')
            print('-' * 30)

        # Confusion Matrix
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.to(device)
        model.eval()

        all_preds, all_true = [], []
        with torch.no_grad():
            for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                outputs = model(X_batch)
                preds   = torch.max(outputs, 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - Combined Speeds - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')
        plt.close()

        writer.close()

        experiment_results.append({
            'config'         : config,
            'final_train_acc': training_acc[-1],
            'final_test_acc' : testing_acc[-1],
            'final_test_loss': testing_loss[-1],
            'best_model_path': best_model_path,
            'history': {
                'train_loss': training_loss,
                'train_acc' : training_acc,
                'test_loss' : testing_loss,
                'test_acc'  : testing_acc
            }
        })

    print('\n' + '='*40)
    print('HYPERPARAMETER TUNING RESULTS - COMBINED')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']}")
        print(f"  -> Final Test Acc: {res['final_test_acc']:.4f}")
        print(f"  -> Final Test Loss: {res['final_test_loss']:.4f}")
        print('-' * 20)

    return experiment_results


print('\n' + '#'*50)
print('TRAINING - COMBINED SPEEDS (1.83 Hz + 5.10 Hz)')
print('#'*50)
results = run_experiments(train_dataset, test_dataset, X_test, y_test)

In [ ]:
# Analyze Hyperparameter Tuning Results
results_df = pd.DataFrame([
    {
        'batch_size'     : res['config']['batch_size'],
        'lr'             : res['config']['lr'],
        'weight_decay'   : res['config']['weight_decay'],
        'final_test_acc' : res['final_test_acc'],
        'final_test_loss': res['final_test_loss']
    }
    for res in results
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Hyperparameter Impact on Test Accuracy - Combined Speeds', fontsize=16)

lrs = sorted(results_df['lr'].unique())
axes[0].boxplot([results_df[results_df['lr'] == lr]['final_test_acc'] for lr in lrs], labels=lrs)
axes[0].set_title('Learning Rate vs Test Accuracy')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')

wds = sorted(results_df['weight_decay'].unique())
axes[1].boxplot([results_df[results_df['weight_decay'] == wd]['final_test_acc'] for wd in wds], labels=wds)
axes[1].set_title('Weight Decay vs Test Accuracy')
axes[1].set_xlabel('Weight Decay')
axes[1].set_ylabel('Test Accuracy')

plt.tight_layout()
_RES = 'results_1d_cnn_combined_speed_csv'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'hyperparam_impact_combined.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] hyperparam_impact_combined.png -> {_p}')
plt.show()

plt.figure(figsize=(12, 6))
for res in results:
    config_str = f"LR={res['config']['lr']}, WD={res['config']['weight_decay']}"
    plt.plot(res['history']['test_acc'], label=config_str)
plt.title('Test Accuracy History - Combined Speeds')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
_RES = 'results_1d_cnn_combined_speed_csv'
import os as _os; _os.makedirs(_RES, exist_ok=True)
_p = _os.path.join(_RES, 'accuracy_history_combined.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
print(f'[SAVED] accuracy_history_combined.png -> {_p}')
plt.show()

In [ ]:
best = max(results, key=lambda x: x['final_test_acc'])

plt.figure(figsize=(7, 4))
plt.plot(best['history']['train_loss'], color='blue', label='Training loss')
plt.plot(best['history']['test_loss'],  color='red',  label='Testing loss')
plt.title('Loss Curves - Combined Speeds (Best Config)')
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(best['history']['train_acc'], color='blue', label='Training acc')
plt.plot(best['history']['test_acc'],  color='red',  label='Testing acc')
plt.title('Accuracy Curves - Combined Speeds (Best Config)')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred = [], []
best        = max(results, key=lambda x: x['final_test_acc'])
test_loader = DataLoader(test_dataset, batch_size=2048, shuffle=False, num_workers=0)

model = Model().to(device)
model.load_state_dict(torch.load(best['best_model_path'], map_location=device))
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs      = model(X_batch.to(device))
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.squeeze().tolist())
        y_pred.extend(predicted.cpu().tolist())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.title('Confusion Matrix - Combined Speeds (Best Config)')
plt.show()

print(f"\nBest Config: {best['config']}")
print(f"Best Test Accuracy: {best['final_test_acc']:.4f}")

In [ ]:
# Save best model
import json

if results:
    best_result = max(results, key=lambda x: x['final_test_acc'])
    best_config = best_result['config']
    print(f"Best Config: {best_config} with Acc: {best_result['final_test_acc']:.4f}")

    print('Retraining best model on full training set...')
    full_train_dataset = RockDataset(X_train, y_train)
    train_loader = DataLoader(full_train_dataset, batch_size=best_config['batch_size'], shuffle=True, num_workers=0)

    model = Model().to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=best_config['lr'], weight_decay=best_config['weight_decay'])
    criterion = nn.CrossEntropyLoss()

    for epoch in range(best_config['epochs']):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.squeeze().long().to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
    print('Retraining complete.')

    os.makedirs('results_1d_cnn_combined_speed_csv', exist_ok=True)
    torch.save(model.state_dict(), 'results_1d_cnn_combined_speed_csv/1d_cnn_combined_speeds.pth')
    with open('results_1d_cnn_combined_speed_csv/1d_cnn_combined_speeds_class_names.json', 'w') as f:
        json.dump(class_names, f)
    print('Saved 1d_cnn_combined_speeds.pth and 1d_cnn_combined_speeds_class_names.json')

In [ ]:
print('To view TensorBoard, run this command in your terminal:')
print()
print('  cd ~/results_1d_cnn_combined_speed_csv && tensorboard --logdir=.')
print()
print('Then open: http://localhost:6006')